# Notebook 11 · Preprocessing figure without titles inside the image

**What it does:** redraws the preprocessing figure of the article (cell 8 of notebook 04) without titles
inside the image; that information goes to the figure caption. The examination (CQ500-CT-173), the rule for
the slice (the one with the most BHX boxes) and the functions for the head centre and the crop are those of
notebook 04, copied without change. The right panel is the PNG the models received (read from
`cq500_512.zip`).

**Before running:** nothing beyond the data already on the Drive. No GPU is needed (*CPU*). **Time:** about
1 minute.

**Language:** documentation and comments are in English; identifiers, printed messages and the column
names of the files read here are in Portuguese, exactly as in the run that produced the results.

In [ ]:
# 1. Drive, libraries and parameters (the same as in notebook 04)
import csv, io, os, subprocess, sys, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DADOS = Path('/content/drive/MyDrive/ich-benchmark/dados')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pydicom', 'pylibjpeg', 'pylibjpeg-libjpeg'],
                   check=True)
except ImportError:                                  # local test: `DADOS` mimics the Drive folder
    DADOS = Path(os.environ['PREPARO_LOCAL'])

import numpy as np
import pydicom
from scipy import ndimage
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

CQ500, PREPARO = DADOS / 'tc' / 'cq500', DADOS / 'tc' / 'preparo'
PIXEL_MM = 0.488281
LADO = 512
JANELAS = {'brain': (40, 80), 'subdural': (80, 200), 'bone': (600, 2800)}
CABECA_HU = -300
EXEMPLO = 'CQ500-CT-173'

In [ ]:
# 2. Functions of notebook 04, unchanged
def janelas(hu):
    """HU (H × W) → 3 canais de 0 a 1 (cérebro, subdural, osso)."""
    return np.stack([np.clip((hu - (n - l / 2)) / l, 0, 1) for n, l in JANELAS.values()], axis=-1)

def centro_da_cabeca(volume):
    """Centro (linha, coluna) e tamanho (altura, largura) da cabeça, em pixels da imagem original.
    Em cada corte (numa cópia reduzida 4×, por velocidade): tecido acima de −300 HU, sem estruturas finas
    (apoio de cabeça, mesa), maior região. Centro = mediana dos centros dos cortes; tamanho = o maior."""
    centros, tamanhos = [], []
    for corte in volume[:, ::4, ::4]:
        tecido = ndimage.binary_opening(corte > CABECA_HU, structure=np.ones((4, 4)))
        rotulos, n = ndimage.label(tecido)
        if n == 0:
            continue
        maior = 1 + int(np.argmax(ndimage.sum(tecido, rotulos, range(1, n + 1))))
        linhas, colunas = np.nonzero(rotulos == maior)
        centros.append(((linhas.min() + linhas.max() + 1) * 2, (colunas.min() + colunas.max() + 1) * 2))
        tamanhos.append(((linhas.max() + 1 - linhas.min()) * 4, (colunas.max() + 1 - colunas.min()) * 4))
    if not centros:
        return (volume.shape[1] / 2, volume.shape[2] / 2), (0, 0)
    c = np.median(np.array(centros, dtype=float), axis=0)
    t = np.array(tamanhos).max(axis=0)
    return (float(c[0]), float(c[1])), (int(t[0]), int(t[1]))

def geometria(pixel_mm, centro):
    """Escala e deslocamento: coordenada contínua de saída = entrada × escala − deslocamento."""
    escala = pixel_mm / PIXEL_MM
    return escala, (centro[0] * escala - LADO / 2, centro[1] * escala - LADO / 2)

In [ ]:
# 3. Original slice (selected series), BHX boxes and the PNG the models received
e = {l['Name']: l for l in csv.DictReader(open(PREPARO / 'cq500_series_escolhidas.csv'))}[EXEMPLO]
caixas = [l for l in csv.DictReader(open(PREPARO / 'cq500_caixas.csv')) if l['Name'] == EXEMPLO]
recuperado = CQ500 / 'zips_recuperados' / f'{EXEMPLO}.zip'
with zipfile.ZipFile(recuperado if recuperado.exists() else CQ500 / 'zips' / f'{EXEMPLO}.zip') as z:
    dcms = sorted((pydicom.dcmread(io.BytesIO(z.read(m))) for m in z.namelist()
                   if m.lower().endswith('.dcm') and m.rsplit('/', 2)[-2] == e['pasta']),
                  key=lambda d: float(d.ImagePositionPatient[2]))
vol = np.stack([d.pixel_array * float(d.RescaleSlope) + float(d.RescaleIntercept) for d in dcms]).astype(np.float32)
pixel = float(dcms[0].PixelSpacing[0])
k = max(range(len(vol)), key=lambda i: sum(int(c['corte']) == i for c in caixas))   # slice with the most boxes
centro, _ = centro_da_cabeca(vol)
escala, desloc = geometria(pixel, centro)
with zipfile.ZipFile(PREPARO / 'cq500_512.zip') as z:
    depois = np.asarray(Image.open(io.BytesIO(z.read(f'images/{EXEMPLO}_s{k:03d}.png'))))
    caixas_512 = [l for l in csv.DictReader(io.TextIOWrapper(z.open('boxes_512.csv'), 'utf-8'))
                  if l['exame'] == EXEMPLO and int(l['corte']) == k]
print(f'{EXEMPLO}: corte {k + 1} de {len(vol)}; {pixel:.6f} mm/pixel; '
      f'{sum(int(c["corte"]) == k for c in caixas)} caixas na BHX, {len(caixas_512)} na imagem de 512')

In [ ]:
# 4. Figure without titles inside the image (the information goes to the caption)
fig, eixos = plt.subplots(1, 2, figsize=(10, 5))
eixos[0].imshow(janelas(vol[k])[..., 0], cmap='gray')
for c in caixas:
    if int(c['corte']) == k:
        eixos[0].add_patch(Rectangle((float(c['x']), float(c['y'])), float(c['largura']), float(c['altura']),
                                     fill=False, color='yellow', lw=1))
lado = LADO / escala                                   # 25 cm crop drawn on the original image
eixos[0].add_patch(Rectangle((desloc[1] / escala, desloc[0] / escala), lado, lado, fill=False, color='cyan',
                             ls='--', lw=1.2))
eixos[1].imshow(depois)
for c in caixas_512:
    x1, y1, x2, y2 = (float(c[v]) for v in ('x1', 'y1', 'x2', 'y2'))
    eixos[1].add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='yellow', lw=1))
for ax in eixos:
    ax.axis('off')
fig.tight_layout(w_pad=1.0)
for ext in ('png', 'pdf'):
    fig.savefig(PREPARO / f'f3-04-preprocessing-notitle.{ext}', dpi=200, bbox_inches='tight',
                metadata={'CreationDate': None} if ext == 'pdf' else None)
print('gravado em', PREPARO / 'f3-04-preprocessing-notitle.pdf')

## After running

`dados/tc/preparo/f3-04-preprocessing-notitle.pdf` on the Drive is the preprocessing figure of the article.